# LeetCode 1228 — Missing Number In Arithmetic Progression bằng Python

**Link đề:** https://leetcode.com/problems/missing-number-in-arithmetic-progression/
**Độ khó:** Easy · **Chủ đề:** Array, Math

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def missingNumber(self, arr: List[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **cấp số cộng (arithmetic progression, AP)**: số hạng tổng quát `a + i·d`, tổng `(đầu + cuối)·số_số_hạng / 2`.
2. Biết suy ra **công sai** `d` từ phần tử đầu, cuối và **độ dài ban đầu** (`n + 1`, không phải `n`).
3. Đi từ quét tuyến tính → công thức tổng `O(n)` → **binary search** `O(log n)`.
4. Xử lý edge case: dãy **giảm dần**, công sai **bằng 0**, phép chia `//` với số âm.
5. Trace, chứng minh, test đối chiếu, bài tập TODO có đáp án ở cuối.

## 1. Đề bài và ví dụ

Trong mảng `arr`, các giá trị **ban đầu** tạo thành một cấp số cộng: `arr[i+1] - arr[i]` bằng nhau với mọi `i`.
Sau đó **một** giá trị bị xoá khỏi `arr`, và giá trị đó **không phải phần tử đầu tiên hay cuối cùng**.

Cho `arr` (sau khi đã xoá), trả về **giá trị bị xoá**.

| `arr` | Kết quả | Dãy ban đầu |
|---|:---:|---|
| `[5, 7, 11, 13]` | `9` | `[5, 7, 9, 11, 13]` (d = 2) |
| `[15, 13, 12]` | `14` | `[15, 14, 13, 12]` (d = −1) |

**Constraints (theo hiểu biết):** `3 <= arr.length <= 1000`; `0 <= arr[i] <= 10^5`; đề bảo đảm `arr` là một mảng hợp lệ.

Hai điều đáng chú ý: dãy có thể **giảm dần** (d âm), và có thể **mọi phần tử bằng nhau** (d = 0, ví dụ `[1, 1, 1]` → bị xoá là `1`).

## 2. Nền tảng: cấp số cộng và công cụ Python

### 2.1. Cấp số cộng (AP)

Dãy `a, a + d, a + 2d, ..., a + k·d`: số hạng thứ `i` (đếm từ 0) là `a + i·d`. `d` gọi là **công sai**.

In [1]:
a, d = 5, 2
print([a + i * d for i in range(5)])
a, d = 15, -1
print([a + i * d for i in range(4)])
print("range(5, 14, 2) =", list(range(5, 14, 2)))   # range tự sinh AP khi d != 0

[5, 7, 9, 11, 13]
[15, 14, 13, 12]
range(5, 14, 2) = [5, 7, 9, 11, 13]


### 2.2. Suy ra công sai từ đầu và cuối

Dãy ban đầu có `m` phần tử thì `cuối - đầu = (m - 1)·d`. Ở bài này `arr` còn `n` phần tử, ban đầu có `m = n + 1` ⇒

```text
d = (arr[-1] - arr[0]) / n
```

Phần tử đầu và cuối **không bị xoá** nên `arr[0]`, `arr[-1]` đúng là đầu, cuối của dãy gốc.

In [2]:
arr = [5, 7, 11, 13]
n = len(arr)
print("arr[0] =", arr[0], "| arr[-1] =", arr[-1], "| n =", n)
print("d =", (arr[-1] - arr[0]) // n)

arr[0] = 5 | arr[-1] = 13 | n = 4
d = 2


### 2.3. `//` với số âm

`//` làm tròn **xuống** (về −∞): `-3 // 2 = -2`, không phải `-1`. Ở bài này `arr[-1] - arr[0]` **luôn chia hết** cho `n` (vì bằng `n·d`), nên `//` cho kết quả chính xác, kể cả khi âm.

In [3]:
print("-3 // 2 =", -3 // 2, "| -3 / 2 =", -3 / 2, "| int(-3 / 2) =", int(-3 / 2))
arr = [15, 13, 12]
print("d =", (arr[-1] - arr[0]) // len(arr), "(chia hết nên chính xác)")

-3 // 2 = -2 | -3 / 2 = -1.5 | int(-3 / 2) = -1
d = -1 (chia hết nên chính xác)


### 2.4. Tổng của cấp số cộng

Tổng `m` số hạng = `(đầu + cuối) · m / 2` — ghép cặp đầu–cuối, mỗi cặp có cùng tổng (mẹo của Gauss).

In [4]:
seq = [5, 7, 9, 11, 13]
print("sum thật     =", sum(seq))
print("công thức    =", (seq[0] + seq[-1]) * len(seq) // 2)
print("cặp đầu-cuối:", [(seq[i], seq[-1 - i], seq[i] + seq[-1 - i]) for i in range(len(seq) // 2)])

sum thật     = 45
công thức    = 45
cặp đầu-cuối: [(5, 13, 18), (7, 11, 18)]


`(đầu + cuối)·m` luôn chẵn với dãy số nguyên AP (một trong hai thừa số chẵn), nên dùng `// 2` an toàn.

### 2.5. Binary search thủ công (`lo`, `hi`, `mid`)

Tìm vị trí đầu tiên thoả một điều kiện **đơn điệu** (sai…sai, đúng…đúng): mỗi bước xét `mid = (lo + hi) // 2` và loại một nửa.

In [5]:
flags = [False, False, False, True, True]     # tìm vị trí True đầu tiên
lo, hi = 0, len(flags) - 1
while lo < hi:
    mid = (lo + hi) // 2
    print(f"lo={lo} hi={hi} mid={mid} flags[mid]={flags[mid]}")
    if flags[mid]:
        hi = mid          # đáp án ở mid hoặc bên trái
    else:
        lo = mid + 1      # đáp án ở bên phải mid
print("vị trí True đầu tiên:", lo)

lo=0 hi=4 mid=2 flags[mid]=False
lo=3 hi=4 mid=3 flags[mid]=True
vị trí True đầu tiên: 3


## 3. Cách 1 — Trực giác: so sánh các hiệu kề nhau

Ý nghĩ tự nhiên: các hiệu `arr[i+1] - arr[i]` đều bằng `d`, **trừ một chỗ** bằng `2d` (chỗ bị xoá). Tìm chỗ "nhảy gấp đôi" đó.

Nhưng ta chưa biết `d`! Cách đơn giản: hiệu có **giá trị tuyệt đối nhỏ hơn** là `d` thật.

In [6]:
def missing_by_diffs(arr: list[int]) -> int:
    diffs = [arr[i + 1] - arr[i] for i in range(len(arr) - 1)]
    d = min(diffs, key=abs)                    # hiệu có |.| nhỏ nhất
    for i in range(len(arr) - 1):
        if arr[i + 1] - arr[i] != d:
            return arr[i] + d
    return arr[0]                              # mọi hiệu bằng nhau -> chỉ xảy ra khi d == 0

for arr in [[5, 7, 11, 13], [15, 13, 12], [1, 1, 1]]:
    print(arr, "->", missing_by_diffs(arr))

[5, 7, 11, 13] -> 9
[15, 13, 12] -> 14
[1, 1, 1] -> 1


`min(diffs, key=abs)` — tìm phần tử có `abs(...)` nhỏ nhất (tham số `key` cho biết so sánh theo tiêu chí nào), nhưng trả về **chính phần tử** (giữ dấu).

Cách này đúng và `O(n)`, nhưng việc "đoán `d` bằng min theo trị tuyệt đối" khá mẹo và dễ sai khi chưa nghĩ kỹ. Có cách **suy ra `d` chắc chắn** từ đầu/cuối.

## 4. Nút thắt và insight

1. **`d` biết chính xác:** `d = (arr[-1] - arr[0]) / n` (mục 2.2). Không cần đoán.
2. **Biết `d` ⇒ biết dãy gốc:** phần tử thứ `i` lẽ ra là `arr[0] + i·d`. Chỗ đầu tiên `arr[i]` lệch khỏi giá trị đó là chỗ bị xoá.
3. **Tổng:** tổng dãy gốc `= (arr[0] + arr[-1])·(n+1)/2`; trừ `sum(arr)` ra đúng số bị xoá — không cần tìm vị trí.
4. **Đơn điệu ⇒ binary search:** trước chỗ xoá `arr[i] == arr[0] + i·d` (khớp), từ chỗ xoá trở đi `arr[i] == arr[0] + (i+1)·d` (lệch). Điều kiện "lệch" có dạng sai…sai, đúng…đúng → tìm nhị phân `O(log n)`.

In [7]:
arr = [5, 7, 11, 13]
d = (arr[-1] - arr[0]) // len(arr)
for i, x in enumerate(arr):
    expected = arr[0] + i * d
    print(f"i={i}: arr[i]={x:2d}, lẽ ra={expected:2d} -> {'khớp' if x == expected else 'LỆCH'}")

i=0: arr[i]= 5, lẽ ra= 5 -> khớp
i=1: arr[i]= 7, lẽ ra= 7 -> khớp
i=2: arr[i]=11, lẽ ra= 9 -> LỆCH
i=3: arr[i]=13, lẽ ra=11 -> LỆCH


## 5. Lời giải chính — giảng từng đoạn rồi ghép

### Cách 2 — Quét tuyến tính với `d` chính xác

**Đoạn 1:** `d = (arr[-1] - arr[0]) // n`.
**Đoạn 2:** duyệt `i`, nếu `arr[i] != arr[0] + i*d` thì trả `arr[0] + i*d`.
**Đoạn 3:** nếu không lệch ở đâu (chỉ khi `d == 0`) thì trả `arr[0]`.

In [8]:
def missing_linear(arr: list[int]) -> int:
    n = len(arr)
    d = (arr[-1] - arr[0]) // n              # đoạn 1
    for i in range(n):                       # đoạn 2
        expected = arr[0] + i * d
        if arr[i] != expected:
            return expected
    return arr[0]                            # đoạn 3 (d == 0)

for arr in [[5, 7, 11, 13], [15, 13, 12], [1, 1, 1]]:
    print(arr, "->", missing_linear(arr))

[5, 7, 11, 13] -> 9
[15, 13, 12] -> 14
[1, 1, 1] -> 1


### Cách 3 — Công thức tổng

Tổng dãy gốc (`n + 1` số hạng) trừ tổng hiện có:

In [9]:
def missing_sum(arr: list[int]) -> int:
    n = len(arr)
    full_sum = (arr[0] + arr[-1]) * (n + 1) // 2
    return full_sum - sum(arr)

for arr in [[5, 7, 11, 13], [15, 13, 12], [1, 1, 1]]:
    print(arr, "-> tổng gốc", (arr[0] + arr[-1]) * (len(arr) + 1) // 2, "- tổng hiện có", sum(arr), "=", missing_sum(arr))

[5, 7, 11, 13] -> tổng gốc 45 - tổng hiện có 36 = 9
[15, 13, 12] -> tổng gốc 54 - tổng hiện có 40 = 14
[1, 1, 1] -> tổng gốc 4 - tổng hiện có 3 = 1


Đẹp: trường hợp `d = 0` **tự động đúng** (`(1+1)·4/2 - 3 = 1`), không cần nhánh riêng.

### Cách 4 — Binary search `O(log n)`

Tìm chỉ số **nhỏ nhất** `i` mà `arr[i] != arr[0] + i*d`. Chỉ số đó tồn tại khi `d != 0` (ít nhất `i = n - 1` lệch, vì `arr[-1] = arr[0] + n·d`).

In [10]:
def missing_binary(arr: list[int]) -> int:
    n = len(arr)
    d = (arr[-1] - arr[0]) // n
    if d == 0:
        return arr[0]
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] == arr[0] + mid * d:     # còn khớp -> chỗ xoá ở bên phải
            lo = mid + 1
        else:                                # đã lệch -> chỗ xoá ở mid hoặc bên trái
            hi = mid
    return arr[0] + lo * d

for arr in [[5, 7, 11, 13], [15, 13, 12], [1, 1, 1]]:
    print(arr, "->", missing_binary(arr))

[5, 7, 11, 13] -> 9
[15, 13, 12] -> 14
[1, 1, 1] -> 1


## 6. TRACE từng bước

### 6.1. Trace binary search trên `arr = [1, 4, 7, 10, 13, 19, 22]` (gốc: 1, 4, ..., 22, d = 3, thiếu 16)

`n = 7`, `d = (22 - 1) // 7 = 3`.

| lo | hi | mid | arr[mid] | lẽ ra `1 + 3·mid` | Khớp? | Hành động |
|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 0 | 6 | 3 | 10 | 10 | ✓ | lo = 4 |
| 4 | 6 | 5 | 19 | 16 | ✗ | hi = 5 |
| 4 | 5 | 4 | 13 | 13 | ✓ | lo = 5 |
| 5 | 5 | — | — | — | — | dừng, trả `1 + 3·5 = 16` |

In [11]:
def trace_missing_binary(arr: list[int]) -> int:
    n = len(arr)
    d = (arr[-1] - arr[0]) // n
    print(f"n={n}, d=({arr[-1]} - {arr[0]}) // {n} = {d}")
    if d == 0:
        print("d == 0 -> mọi phần tử bằng nhau, trả", arr[0])
        return arr[0]
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        exp = arr[0] + mid * d
        ok = arr[mid] == exp
        print(f"lo={lo} hi={hi} mid={mid} arr[mid]={arr[mid]} lẽ ra={exp} -> {'khớp, lo=mid+1' if ok else 'lệch, hi=mid'}")
        if ok:
            lo = mid + 1
        else:
            hi = mid
    ans = arr[0] + lo * d
    print(f"dừng lo=hi={lo} -> số bị xoá = {arr[0]} + {lo}*{d} = {ans}")
    return ans

assert trace_missing_binary([1, 4, 7, 10, 13, 19, 22]) == 16

n=7, d=(22 - 1) // 7 = 3
lo=0 hi=6 mid=3 arr[mid]=10 lẽ ra=10 -> khớp, lo=mid+1
lo=4 hi=6 mid=5 arr[mid]=19 lẽ ra=16 -> lệch, hi=mid
lo=4 hi=5 mid=4 arr[mid]=13 lẽ ra=13 -> khớp, lo=mid+1
dừng lo=hi=5 -> số bị xoá = 1 + 5*3 = 16


### 6.2. Trace dãy giảm và dãy hằng

In [12]:
assert trace_missing_binary([15, 13, 12]) == 14
print()
assert trace_missing_binary([1, 1, 1]) == 1

n=3, d=(12 - 15) // 3 = -1
lo=0 hi=2 mid=1 arr[mid]=13 lẽ ra=14 -> lệch, hi=mid
lo=0 hi=1 mid=0 arr[mid]=15 lẽ ra=15 -> khớp, lo=mid+1
dừng lo=hi=1 -> số bị xoá = 15 + 1*-1 = 14

n=3, d=(1 - 1) // 3 = 0
d == 0 -> mọi phần tử bằng nhau, trả 1


### 6.3. Trace quét tuyến tính (thấy rõ "trước khớp, sau lệch")

In [13]:
def trace_linear(arr):
    d = (arr[-1] - arr[0]) // len(arr)
    for i, x in enumerate(arr):
        exp = arr[0] + i * d
        print(f"i={i} arr[i]={x} lẽ ra={exp} {'' if x == exp else '<- lệch đầu tiên, trả ' + str(exp)}")
        if x != exp:
            return exp
    return arr[0]

assert trace_linear([5, 7, 11, 13]) == 9

i=0 arr[i]=5 lẽ ra=5 
i=1 arr[i]=7 lẽ ra=7 
i=2 arr[i]=11 lẽ ra=9 <- lệch đầu tiên, trả 9


## 7. Code nộp LeetCode

### 7.1. Bản chính — công thức tổng (ngắn, tự xử lý d = 0)

In [14]:
from typing import List

class Solution:
    def missingNumber(self, arr: List[int]) -> int:
        n = len(arr)
        return (arr[0] + arr[-1]) * (n + 1) // 2 - sum(arr)

sol = Solution()
print(sol.missingNumber([5, 7, 11, 13]), sol.missingNumber([15, 13, 12]), sol.missingNumber([1, 1, 1]))

9 14 1


### 7.2. Bản binary search

In [15]:
class SolutionBinary:
    def missingNumber(self, arr: List[int]) -> int:
        n = len(arr)
        d = (arr[-1] - arr[0]) // n
        if d == 0:
            return arr[0]
        lo, hi = 0, n - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if arr[mid] == arr[0] + mid * d:
                lo = mid + 1
            else:
                hi = mid
        return arr[0] + lo * d

assert SolutionBinary().missingNumber([15, 13, 12]) == 14
print("SolutionBinary OK")

SolutionBinary OK


### 7.3. Bản quét tuyến tính

In [16]:
class SolutionLinear:
    def missingNumber(self, arr: List[int]) -> int:
        d = (arr[-1] - arr[0]) // len(arr)
        for i, x in enumerate(arr):
            if x != arr[0] + i * d:
                return arr[0] + i * d
        return arr[0]

assert SolutionLinear().missingNumber([5, 7, 11, 13]) == 9
print("SolutionLinear OK")

SolutionLinear OK


## 8. Vì sao đúng?

Gọi dãy gốc `g_0, ..., g_n` với `g_i = a + i·d`, phần tử bị xoá là `g_k` với `1 ≤ k ≤ n - 1`. Khi đó `arr = [g_0, ..., g_{k-1}, g_{k+1}, ..., g_n]`, có `n` phần tử.

- **Công sai:** `arr[0] = g_0 = a`, `arr[-1] = g_n = a + n·d` ⇒ `d = (arr[-1] - arr[0]) / n` (chia hết).
- **Quét tuyến tính:** với `i < k`: `arr[i] = g_i = a + i·d` (khớp). Với `i = k`: `arr[k] = g_{k+1} = a + (k+1)·d ≠ a + k·d` khi `d ≠ 0` (lệch). Vậy chỉ số lệch đầu tiên là `k`, và giá trị "lẽ ra" ở đó là `g_k` — đúng số bị xoá.
- **Binary search:** vị từ `P(i) = "arr[i] lệch"` là sai với `i < k`, đúng với `i ≥ k` (với `i ≥ k`, `arr[i] = a + (i+1)·d ≠ a + i·d`). Vị từ đơn điệu ⇒ binary search tìm đúng `k`. Bất biến: `k ∈ [lo, hi]`.
- **Tổng:** `sum(g) = (g_0 + g_n)·(n+1)/2`; `sum(arr) = sum(g) - g_k` ⇒ `g_k = sum(g) - sum(arr)`.
- **d = 0:** mọi phần tử bằng `a`, số bị xoá cũng là `a`; công thức tổng cho `a·(n+1) - a·n = a` ✓. ∎

## 9. Độ phức tạp và so sánh

Gọi `n = len(arr)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Hiệu kề nhau + đoán `d` | `O(n)` | `O(n)` (list diffs) | mẹo `min(key=abs)` |
| Quét tuyến tính với `d` chính xác | `O(n)` | `O(1)` | rõ ràng |
| Công thức tổng | `O(n)` (vì `sum`) | `O(1)` | ngắn nhất, tự xử lý `d = 0` |
| **Binary search** | `O(log n)` | `O(1)` | nhanh nhất về lý thuyết |

Với `n ≤ 1000` cả bốn đều tức thì; binary search là điểm cộng khi phỏng vấn hỏi "làm nhanh hơn `O(n)` được không?".

## 10. Bộ test

Sinh dữ liệu **từ định nghĩa**: chọn `a`, `d`, độ dài, xoá một phần tử giữa ngẫu nhiên → biết chắc đáp án. Bao gồm dãy tăng, giảm, hằng, độ dài tối thiểu.

In [17]:
impls = {
    "diffs": missing_by_diffs,
    "linear": missing_linear,
    "sum": missing_sum,
    "binary": missing_binary,
    "Solution": Solution().missingNumber,
    "SolBinary": SolutionBinary().missingNumber,
    "SolLinear": SolutionLinear().missingNumber,
}
cases = [
    ([5, 7, 11, 13], 9),
    ([15, 13, 12], 14),
    ([1, 1, 1], 1),                       # d = 0
    ([0, 2, 3], 1),                       # độ dài tối thiểu, xoá ở chỉ số 1
    ([0, 1, 3], 2),                       # xoá ở áp chót
    ([100000, 75000, 25000, 0], 50000),   # giảm, giá trị biên
    ([1, 4, 7, 10, 13, 19, 22], 16),
    ([10, 7, 1], 4),
]
for arr, expected in cases:
    for name, f in impls.items():
        assert f(arr) == expected, (name, arr, f(arr))
    print(f"{str(arr):32} -> {expected} PASS")

[5, 7, 11, 13]                   -> 9 PASS
[15, 13, 12]                     -> 14 PASS
[1, 1, 1]                        -> 1 PASS
[0, 2, 3]                        -> 1 PASS
[0, 1, 3]                        -> 2 PASS
[100000, 75000, 25000, 0]        -> 50000 PASS
[1, 4, 7, 10, 13, 19, 22]        -> 16 PASS
[10, 7, 1]                       -> 4 PASS


In [18]:
import random

rng = random.Random(1228)
count = 0
for _ in range(1000):
    m = rng.randint(4, 40)                  # độ dài dãy gốc (>= 4 để arr còn >= 3)
    d = rng.randint(-50, 50)
    a = rng.randint(0, 10**5)
    full = [a + i * d for i in range(m)]
    if min(full) < 0 or max(full) > 10**5:
        continue                            # giữ đúng miền giá trị của đề
    k = rng.randint(1, m - 2)               # không xoá đầu/cuối
    arr = full[:k] + full[k + 1:]
    for name, f in impls.items():
        assert f(arr) == full[k], (name, arr, full[k])
    count += 1
print(f"{count} test sinh từ định nghĩa: mọi cách đều đúng")

995 test sinh từ định nghĩa: mọi cách đều đúng


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chia cho `n - 1` thay vì `n`

In [19]:
def wrong_divide(arr):
    d = (arr[-1] - arr[0]) // (len(arr) - 1)      # SAI: coi arr là dãy gốc
    for i, x in enumerate(arr):
        if x != arr[0] + i * d:
            return arr[0] + i * d
    return arr[0]

print("sai :", wrong_divide([0, 10, 15]), "| đúng:", missing_sum([0, 10, 15]))

sai : 7 | đúng: 5


Với `[0, 10, 15]` (gốc `0, 5, 10, 15`), chia cho `n - 1 = 2` được `d = 7` → sai hoàn toàn. Vì sao: dãy gốc có `n + 1` phần tử ⇒ có `n` khoảng. `arr` đã mất một phần tử nhưng đầu/cuối giữ nguyên, nên số khoảng vẫn là `n`.

### Lỗi 2: Dùng `arr[1] - arr[0]` làm công sai

Nếu phần tử bị xoá là `g_1` thì `arr[1] - arr[0] = 2d` → sai.

In [20]:
def wrong_first_diff(arr):
    d = arr[1] - arr[0]
    for i, x in enumerate(arr):
        if x != arr[0] + i * d:
            return arr[0] + i * d
    return arr[0]

print("sai :", wrong_first_diff([0, 2, 3]), "| đúng:", missing_sum([0, 2, 3]))

sai : 4 | đúng: 1


### Lỗi 3: Quên trường hợp `d = 0` trong quét tuyến tính / binary search

Không có nhánh `return arr[0]` cuối hàm → hàm trả `None`.

In [21]:
def wrong_no_zero_case(arr):
    d = (arr[-1] - arr[0]) // len(arr)
    for i, x in enumerate(arr):
        if x != arr[0] + i * d:
            return arr[0] + i * d
    # quên return arr[0]

print("sai :", wrong_no_zero_case([7, 7, 7]), "| đúng:", missing_sum([7, 7, 7]))

sai : None | đúng: 7


### Lỗi 4: Dùng `/` thay `//` và trả số thực

`(arr[-1] - arr[0]) / n` cho `2.0` → kết quả `9.0` thay vì `9`. LeetCode Python có thể vẫn chấp nhận `9.0 == 9`, nhưng kiểu trả về sai và với số lớn float có thể mất chính xác.

### Lỗi 5: Giả sử dãy luôn tăng

Viết binary search kiểu `if arr[mid] > ...` sẽ hỏng với dãy giảm. So sánh **bằng/không bằng** với giá trị lẽ ra (`==`) thì đúng cho cả hai chiều.

## 12. Mẹo nhận dạng pattern

- "Dãy có **quy luật** biết trước, thiếu **một** phần tử" → (1) tính giá trị lẽ ra theo công thức, (2) dùng tổng/XOR, hoặc (3) binary search trên vị từ "đã lệch chưa".
- Có thể tính "lẽ ra phần tử thứ `i` là gì" ⇒ vị từ đơn điệu ⇒ **binary search** `O(log n)`.
- Bài cùng họ: 268 Missing Number (0..n, dùng tổng hoặc XOR), 1539 Kth Missing Positive Number (binary search trên `arr[i] - i - 1`), 1502 Can Make Arithmetic Progression From Sequence, 41 First Missing Positive.

## 13. Biến thể

### 13.1. Missing Number (LeetCode 268) — dãy 0..n thiếu một số (chưa sắp xếp)

Tổng `0 + 1 + ... + n = n(n+1)/2`, hoặc XOR — vì `x ^ x = 0`.

In [22]:
def missing_0_to_n(nums: list[int]) -> int:
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)

def missing_0_to_n_xor(nums: list[int]) -> int:
    x = len(nums)
    for i, v in enumerate(nums):
        x ^= i ^ v
    return x

print(missing_0_to_n([3, 0, 1]), missing_0_to_n_xor([9, 6, 4, 2, 3, 5, 7, 0, 1]))   # 2 8

2 8


### 13.2. Can Make Arithmetic Progression (LeetCode 1502)

Sắp xếp rồi kiểm tra mọi hiệu kề bằng nhau.

In [23]:
def can_make_ap(arr: list[int]) -> bool:
    s = sorted(arr)
    d = s[1] - s[0]
    return all(s[i + 1] - s[i] == d for i in range(len(s) - 1))

print(can_make_ap([3, 5, 1]), can_make_ap([1, 2, 4]))

True False


### 13.3. Kth Missing Positive Number (LeetCode 1539) — binary search trên "số đã thiếu"

`arr` tăng nghiêm ngặt các số dương. Tới vị trí `i`, số lượng số dương bị thiếu là `arr[i] - (i + 1)` — đại lượng **không giảm** → binary search.

In [24]:
def find_kth_positive(arr: list[int], k: int) -> int:
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) < k:
            lo = mid + 1
        else:
            hi = mid
    return lo + k

print(find_kth_positive([2, 3, 4, 7, 11], 5), find_kth_positive([1, 2, 3, 4], 2))   # 9 6

9 6


## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `arr = [20, 16, 8, 4]`. Tính `d`, dãy gốc, số bị xoá.

**Bài 2 (dự đoán output).** `(4 - 20) // 4`, `-7 // 2`, `(20 + 4) * 5 // 2 - sum([20, 16, 8, 4])`.

**Bài 3 (điền TODO).** Hoàn thiện quét tuyến tính.

In [25]:
def exercise_missing(arr: list[int]) -> int:
    n = len(arr)
    # TODO 1: tính d từ arr[0], arr[-1], n
    # TODO 2: duyệt i, nếu arr[i] khác arr[0] + i*d thì trả giá trị lẽ ra
    # TODO 3: không lệch -> trả arr[0]
    pass

# assert exercise_missing([5, 7, 11, 13]) == 9
# assert exercise_missing([1, 1, 1]) == 1

**Bài 4 (viết lại).** Viết binary search từ đầu (nhớ trường hợp `d == 0`).

In [26]:
def exercise_missing_binary(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_missing_binary([1, 4, 7, 10, 13, 19, 22]) == 16
# assert exercise_missing_binary([15, 13, 12]) == 14

**Bài 5 (biến thể).** `arr` **chưa sắp xếp** (dãy gốc là AP nhưng bị xáo trộn, rồi xoá 1 phần tử không phải min/max). Tìm số bị xoá trong `O(n)` không sort.

In [27]:
def exercise_missing_unsorted(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_missing_unsorted([13, 5, 11, 7]) == 9
# assert exercise_missing_unsorted([12, 15, 13]) == 14

**Bài 6 (mini challenge).** Từ dãy AP **bị xoá đúng 2 phần tử** ở giữa (không phải đầu/cuối, không kề nhau hoặc kề nhau đều được). Trả về 2 số bị xoá theo thứ tự trong dãy.

In [28]:
def exercise_missing_two(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_missing_two([1, 3, 7, 11]) == [5, 9]
# assert exercise_missing_two([2, 8, 10]) == [4, 6]

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Dãy giảm | `d = (cuối - đầu) // n` | viết `20, 20+d, 20+2d, ...` |
| 2 | `//` làm tròn xuống | chia hết thì chính xác | công thức tổng `n + 1` số hạng |
| 3 | So với giá trị lẽ ra | `arr[0] + i * d` | `return arr[0]` ở cuối |
| 4 | Vị từ "lệch" đơn điệu | `lo, hi = 0, n - 1` | khớp → `lo = mid + 1`, lệch → `hi = mid` |
| 5 | min, max là đầu/cuối dãy gốc | `d = (max - min) / n` (lấy trị dương) | tổng gốc `(min + max)·(n+1)/2 - sum(arr)` |
| 6 | Dãy gốc có `n + 2` phần tử | `d = (arr[-1] - arr[0]) // (n + 1)` | quét, mỗi lần lệch thì ghi số lẽ ra và dịch "độ lệch" thêm 1 |

## 16. Đáp án tham khảo

**Bài 1:** `d = (4 - 20) // 4 = -4`; dãy gốc `20, 16, 12, 8, 4` → thiếu **12**.

**Bài 2:** `-4`, `-4`, `12`.

In [29]:
def answer_missing(arr):
    n = len(arr)
    d = (arr[-1] - arr[0]) // n
    for i in range(n):
        if arr[i] != arr[0] + i * d:
            return arr[0] + i * d
    return arr[0]

def answer_missing_binary(arr):
    n = len(arr)
    d = (arr[-1] - arr[0]) // n
    if d == 0:
        return arr[0]
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] == arr[0] + mid * d:
            lo = mid + 1
        else:
            hi = mid
    return arr[0] + lo * d

def answer_missing_unsorted(arr):
    lo, hi, n = min(arr), max(arr), len(arr)
    return (lo + hi) * (n + 1) // 2 - sum(arr)

def answer_missing_two(arr):
    n = len(arr)
    d = (arr[-1] - arr[0]) // (n + 1)
    res = []
    shift = 0                                  # số phần tử đã thiếu trước vị trí hiện tại
    for i in range(n):
        while arr[i] != arr[0] + (i + shift) * d:
            res.append(arr[0] + (i + shift) * d)
            shift += 1
    return res

assert answer_missing([20, 16, 8, 4]) == 12
assert ((4 - 20) // 4, -7 // 2, (20 + 4) * 5 // 2 - sum([20, 16, 8, 4])) == (-4, -4, 12)
assert answer_missing([5, 7, 11, 13]) == 9 and answer_missing([1, 1, 1]) == 1
assert answer_missing_binary([1, 4, 7, 10, 13, 19, 22]) == 16
assert answer_missing_binary([15, 13, 12]) == 14
assert answer_missing_unsorted([13, 5, 11, 7]) == 9
assert answer_missing_unsorted([12, 15, 13]) == 14
assert answer_missing_two([1, 3, 7, 11]) == [5, 9]
assert answer_missing_two([2, 8, 10]) == [4, 6]
assert answer_missing_two([12, 9, 0]) == [6, 3]
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
# Cách tổng (gọn nhất):
n = len(arr)
return (arr[0] + arr[-1]) * (n + 1) // 2 - sum(arr)

# Cách binary search:
d = (arr[-1] - arr[0]) // n
if d == 0: return arr[0]
lo, hi = 0, n - 1
while lo < hi:
    mid = (lo + hi) // 2
    if arr[mid] == arr[0] + mid * d: lo = mid + 1
    else: hi = mid
return arr[0] + lo * d
```

**Câu thần chú:** *"Dãy gốc có n + 1 số → n khoảng. Biết d là biết mọi thứ."*

### Checklist trước khi submit
- [ ] Chia cho `n` (= `len(arr)`), không phải `n - 1`?
- [ ] Xử lý `d == 0` (hoặc dùng công thức tổng tự xử lý)?
- [ ] So sánh `==` / `!=` với giá trị lẽ ra (đúng cho cả dãy tăng và giảm)?
- [ ] Dùng `//` để kết quả là `int`?
- [ ] Tên method `missingNumber(self, arr)`?